In [1]:
#--- Ollama LLM Multi-Model Conversation ---
import os
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv()

# Model configuration using dataclass
@dataclass
class ModelConfig:
    name: str
    llm: ChatOllama
    system_prompt: str
    emoji: str
    role: str

# Create LLM instances
def create_llm(model_env_var: str, temperature: float = 0.7) -> ChatOllama:
    return ChatOllama(
        base_url=os.getenv("URL_BASE_OLLAMA"),
        model=os.getenv(model_env_var),
        temperature=temperature
    )

# Centralized model configuration
MODELS = {
    "mistral": ModelConfig(
        name="MISTRAL",
        llm=create_llm("MODEL_NAME_OLLAMA_MISTRAL"),
        system_prompt="""You are an expert economist with extensive knowledge in macroeconomics, 
        financial markets and economic policies. Analyze topics from an economic perspective, 
        considering impacts on employment, productivity and growth. 
        Respond concisely and with solid reasoning.""",
        emoji="🔵",
        role="Economist"
    ),
    "gemma": ModelConfig(
        name="GEMMA",
        llm=create_llm("MODEL_NAME_OLLAMA_GEMMA3"),
        system_prompt="""You are an expert in technology and innovation with deep knowledge in 
        artificial intelligence, software development and digital transformation. 
        Analyze topics from a technological perspective, considering trends, 
        implementation and technical challenges. Respond concisely and with solid reasoning.""",
        emoji="🟢",
        role="Technologist"
    )
}

In [2]:
#--- Chat History Management by Session ---

# Session history store
store: dict[str, ChatMessageHistory] = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """Get or create chat history for a session."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

def clear_session_history(session_id: str) -> None:
    """Clear the history of a specific session."""
    if session_id in store:
        del store[session_id]

def create_chain_with_history(model_config: ModelConfig) -> RunnableWithMessageHistory:
    """Create a chain with history for a given model."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", model_config.system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])
    
    return RunnableWithMessageHistory(
        prompt | model_config.llm,
        get_session_history,
        input_messages_key="input",
        history_messages_key="chat_history",
    )

# Create chains for each model
chains = {key: create_chain_with_history(config) for key, config in MODELS.items()}

# Debate history
debate_history: list[dict] = []

def run_multi_model_conversation(
    topic: str, 
    session_id: str, 
    rounds: int = 3,
    model_order: list[str] = None
) -> list[dict]:
    """
    Run a conversation between models on a topic.
    
    Args:
        topic: Discussion topic
        session_id: Unique session ID
        rounds: Number of exchange rounds (each round = all models speak once)
        model_order: List with the order of models (default: ["mistral", "gemma"])
    
    Returns:
        List with the debate history
    """
    if model_order is None:
        model_order = ["mistral", "gemma"]
    
    config = {"configurable": {"session_id": session_id}}
    debate_history.clear()
    clear_session_history(session_id)
    
    print(f"{'='*60}")
    print(f"🎯 Conversation about: {topic}")
    print(f"📊 Rounds: {rounds} | Participants: {len(model_order)}")
    print(f"{'='*60}\n")
    
    last_response = None
    
    # Each round = all models speak once
    for round_num in range(1, rounds + 1):
        print(f"{'─'*20} Round {round_num}/{rounds} {'─'*20}\n")
        
        for i, model_key in enumerate(model_order):
            model_config = MODELS[model_key]
            
            # Build prompt based on context
            if round_num == 1 and i == 0:
                # First model, first round: start the discussion
                input_text = f"Start a discussion about: {topic}"
            else:
                # Rest: respond to previous
                prev_model_key = model_order[(i - 1) % len(model_order)]
                prev_role = MODELS[prev_model_key].role
                input_text = f"Respond to this {prev_role}'s analysis concisely: {last_response}"
            
            response = chains[model_key].invoke(
                {"input": input_text},
                config=config
            )
            
            print(f"{model_config.emoji} {model_config.name} ({model_config.role}):")
            print(f"{response.content}\n")
            
            debate_history.append({
                "round": round_num,
                "model": model_config.name, 
                "role": model_config.role, 
                "content": response.content
            })
            
            last_response = response.content
    
    # Debate summary
    print(f"\n{'='*60}")
    print("📋 DEBATE SUMMARY")
    print(f"{'='*60}")
    
    current_round = 0
    for entry in debate_history:
        if entry["round"] != current_round:
            current_round = entry["round"]
            print(f"\n🔄 Round {current_round}:")
        
        preview = entry['content'][:100] + '...' if len(entry['content']) > 100 else entry['content']
        print(f"   {MODELS[entry['model'].lower()].emoji} [{entry['model']}]: {preview}")
    
    return debate_history

# Run the conversation
result = run_multi_model_conversation(
    topic="The impact of artificial intelligence on the job market",
    session_id="ai-debate-2024",
    rounds=2
)

🎯 Conversation about: The impact of artificial intelligence on the job market
📊 Rounds: 2 | Participants: 2

──────────────────── Round 1/2 ────────────────────

🔵 MISTRAL (Economist):
 Artificial Intelligence (AI) has the potential to significantly transform various sectors of the economy, leading to both opportunities and challenges in the labor market.

On one hand, AI can automate routine tasks, reducing labor costs and increasing productivity. This is particularly evident in manufacturing, where AI-powered machines can work faster and more accurately than human workers. However, increased productivity may lead to economic growth and higher living standards.

On the other hand, AI could displace jobs as it becomes capable of performing tasks that were previously done by humans. According to a McKinsey Global Institute report, up to 800 million global jobs could be lost to automation by 2030. This is especially true for low-skilled and manual labor roles, such as factory workers or 